In [ ]:
# Shared paths: configure raw data once in config.local.toml at the repo root.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_project = next(
    (p for p in (_start, *_start.parents) if (p / "scripts" / "project_paths.py").is_file()),
    None,
)
if _project is None:
    raise RuntimeError("Start the notebook kernel in the repository or a subdirectory.")
_scripts = str(_project / "scripts")
if _scripts not in sys.path:
    sys.path.insert(0, _scripts)
from project_paths import PROJECT_ROOT, MIMIC_DATA_DIR, PROCESSED_DIR, REPORTS_DIR, mimic_csv


In [1]:
import pandas as pd
from pathlib import Path

processed_path = PROCESSED_DIR

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

In [3]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    cv.split(X_train, y_train, groups=subject_id_train)
)

X_dl_train = X_train.iloc[train_idx]
X_val = X_train.iloc[val_idx]

y_dl_train = y_train.iloc[train_idx]
y_val = y_train.iloc[val_idx]

In [4]:
X_dl_train_processed = preprocessor.fit_transform(X_dl_train)
X_val_processed = preprocessor.transform(X_val)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Dense(64, activation="relu", input_shape=(X_dl_train_processed.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_dl_train_processed,
    y_dl_train,
    validation_data=(X_val_processed, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

I0000 00:00:1789559968.309025    8965 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789559968.317818    8965 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789559968.354627    8965 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789559969.660332    8965 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

Epoch 1/50


/home/sengul/projects/icu-risk-prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789559971.949459    8965 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789559971.950060   14489 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789559971.989105    8965 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if

453/453 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8872 - loss: 0.2979 - val_accuracy: 0.8948 - val_loss: 0.2742
Epoch 2/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8984 - loss: 0.2632 - val_accuracy: 0.8913 - val_loss: 0.2736
Epoch 3/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9020 - loss: 0.2521 - val_accuracy: 0.8972 - val_loss: 0.2659
Epoch 4/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9044 - loss: 0.2441 - val_accuracy: 0.8963 - val_loss: 0.2642
Epoch 5/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9072 - loss: 0.2369 - val_accuracy: 0.8960 - val_loss: 0.2638
Epoch 6/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9095 - loss: 0.2311 - val_accuracy: 0.8959 - val_loss: 0.2635
Epoch 7/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9114 - loss: 0.2263 - val_accuracy: 0.8942 - val_loss: 0.2644
Epoch 8/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9140 - loss: 0.2202 - val_accuracy: 0.8891 - val_

In [8]:
len(history.history["loss"])

11

In [9]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

val_probs = model.predict(X_val_processed).ravel()

val_pred = (val_probs >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, val_probs))
print("Precision:", precision_score(y_val, val_pred))
print("Recall:", recall_score(y_val, val_pred))
print("F1:", f1_score(y_val, val_pred))

227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step
ROC-AUC: 0.8643848409519739
Precision: 0.6244444444444445
Recall: 0.3248554913294798
F1: 0.42737642585551333


In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_dl_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)

{0: np.float64(0.5679543045577671), 1: np.float64(4.17894280762565)}


In [11]:
balanced_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_dl_train_processed.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

balanced_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

balanced_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

balanced_history = balanced_model.fit(
    X_dl_train_processed,
    y_dl_train,
    validation_data=(X_val_processed, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[balanced_early_stopping],
    class_weight=class_weights,
    verbose=1
)

Epoch 1/50


/home/sengul/projects/icu-risk-prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7241 - loss: 0.5230 - val_accuracy: 0.7367 - val_loss: 0.5159
Epoch 2/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7624 - loss: 0.4620 - val_accuracy: 0.7544 - val_loss: 0.4970
Epoch 3/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7726 - loss: 0.4387 - val_accuracy: 0.8030 - val_loss: 0.4101
Epoch 4/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7847 - loss: 0.4242 - val_accuracy: 0.7809 - val_loss: 0.4486
Epoch 5/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7946 - loss: 0.4100 - val_accuracy: 0.7581 - val_loss: 0.4891
Epoch 6/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8019 - loss: 0.3953 - val_accuracy: 0.7571 - val_loss: 0.4871
Epoch 7/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8080 - loss: 0.3846 - val_accuracy: 0.7809 - val_loss: 0.4468
Epoch 8/50
453/453 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8132 - loss: 0.3735 - val_accuracy: 0.7683 - val_

In [12]:
len(balanced_history.history["loss"])

8

In [13]:
balanced_val_probs = balanced_model.predict(X_val_processed).ravel()

balanced_val_pred = (balanced_val_probs >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, balanced_val_probs))
print("Precision:", precision_score(y_val, balanced_val_pred))
print("Recall:", recall_score(y_val, balanced_val_pred))
print("F1:", f1_score(y_val, balanced_val_pred))

227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 937us/step
ROC-AUC: 0.8683327933260545
Precision: 0.34962406015037595
Recall: 0.7526011560693642
F1: 0.47744774477447743


In [14]:
thresholds = [0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:
    balanced_val_pred = (balanced_val_probs >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", precision_score(y_val, balanced_val_pred))
    print("Recall:", recall_score(y_val, balanced_val_pred))
    print("F1:", f1_score(y_val, balanced_val_pred))
    print()

Threshold: 0.5
Precision: 0.34962406015037595
Recall: 0.7526011560693642
F1: 0.47744774477447743

Threshold: 0.6
Precision: 0.40445402298850575
Recall: 0.6508670520231213
F1: 0.49889233495790875

Threshold: 0.7
Precision: 0.48370136698212407
Recall: 0.5317919075144508
F1: 0.5066079295154186

Threshold: 0.8
Precision: 0.5753899480069324
Recall: 0.3838150289017341
F1: 0.4604715672676838



## Deep Learning

A feed-forward neural network was evaluated for ICU mortality prediction.

- Numerical features were processed using median imputation followed by `StandardScaler`.
- Categorical features were transformed using `OneHotEncoder`.
- The training data was split into train and validation subsets using `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same group.
- A neural network with two hidden layers was used:
  - 64 neurons
  - 32 neurons
  - 1 output neuron
- ReLU activation was used in the hidden layers.
- Sigmoid activation was used in the output layer for binary classification.
- The model was trained with:
  - Optimizer: `Adam`
  - Loss: `binary_crossentropy`
  - Batch size: `64`
  - Maximum epochs: `50`
- `EarlyStopping` with `patience=5` and `restore_best_weights=True` was used to reduce overfitting.

### Baseline Neural Network

The baseline model stopped after 11 epochs due to EarlyStopping.

Validation performance:

- ROC-AUC: **0.864**
- Precision: **0.624**
- Recall: **0.325**
- F1: **0.427**

The model showed good discrimination and precision, but recall remained relatively low.

### Class Weight Balancing

Because mortality is the minority class, class weights were added during model training.

The balanced model stopped after 8 epochs due to EarlyStopping.

Validation performance:

- ROC-AUC: **0.868**
- Precision: **0.350**
- Recall: **0.753**
- F1: **0.477**

Class weighting substantially improved recall, although precision decreased.

### Threshold Tuning

Different classification thresholds were evaluated on the balanced neural network:

- Threshold `0.5`: Precision **0.350**, Recall **0.753**, F1 **0.477**
- Threshold `0.6`: Precision **0.404**, Recall **0.651**, F1 **0.499**
- Threshold `0.7`: Precision **0.484**, Recall **0.532**, F1 **0.507**
- Threshold `0.8`: Precision **0.575**, Recall **0.384**, F1 **0.460**

Threshold `0.7` produced the highest F1 score, but threshold `0.6` was preferred because recall was considered more important for ICU mortality prediction.

Overall, the selected Deep Learning configuration was:

- Two hidden layers: `64 → 32`
- Class weighting enabled
- Classification threshold = **0.6**

This configuration provided a stronger recall-focused balance for the clinical objective.